# Explore: entity extraction and grounding

Scratch notebook for running the pipeline from `README.md` against real
transcriptions in `data/mtsamples.csv`, one stage at a time.

1. **Extraction** (`src/extraction.py`) — pull raw entity spans out of a transcription.
2. **Ontology retrieval** (`src/ontology.py`) — fetch a small candidate shortlist per entity from OLS4.
3. **Grounding** (`src/grounding.py`) — Typesafe's Jev model picks the correct candidate, or `NO_MATCH`, from that shortlist.

Stage 4 (dose validation) is not wired up here yet.

Extraction and grounding calls hit paid APIs (OpenAI and Typesafe).
Grounding results are cached to `grounding_cache.json` at the project root,
so re-running this notebook replays cached answers instead of re-spending
on identical calls. Delete that file to force live calls.

In [1]:
import pandas as pd
from tqdm import tqdm
from src.extraction import extract_entities
from src.ontology import search_candidates
from src.grounding import ground_entity

## Load the sample transcriptions

Loads `data/mtsamples.csv` (the public MTSamples dataset) and previews the
last 50 rows. Each row's `transcription` column is a full clinical note;
`d.iloc[<row>]['transcription']` below picks one to run through the
pipeline.

In [2]:
d = pd.read_csv("data/mtsamples.csv")

In [3]:
d.tail(50)

,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
4949,4949,"Bronchiolitis, respiratory syncytial virus po...",Cardiovascular / Pulmonary,Bronchiolitis - Discharge Summary,"DIAGNOSES:,1. Bronchiolitis, respiratory sync...","cardiovascular / pulmonary, bronchiolitis, res..."
4950,4950,Fiberoptic bronchoscopy for diagnosis of righ...,Cardiovascular / Pulmonary,Bronchoscopy - 3,"PROCEDURE: , Fiberoptic bronchoscopy.,PREOPERA...","cardiovascular / pulmonary, bronchoscopy, fibe..."
4951,4951,"Rigid bronchoscopy with dilation, excision of...",Cardiovascular / Pulmonary,Bronchoscopy - 4,"PREOPERATIVE DIAGNOSIS: ,Tracheal stenosis an...","cardiovascular / pulmonary, tracheal stenosis,..."
4952,4952,Bilateral carotid cerebral angiogram and righ...,Cardiovascular / Pulmonary,Bilateral Carotid Cerebral Angiogram,"PREOPERATIVE DIAGNOSES:,1. Carotid artery occ...","cardiovascular / pulmonary, femoral-popliteal ..."
4953,4953,"Bronchoscopy, right upper lobe biopsies and r...",Cardiovascular / Pulmonary,Bronchoscopy - 1,"PROCEDURE:, Bronchoscopy, right upper lobe bi...","cardiovascular / pulmonary, bronchoscopy, wang..."
4954,4954,2-month-old female - increased work of breath...,Cardiovascular / Pulmonary,Bronchiolitis - 2-month-old,"CHIEF COMPLAINT: , Increased work of breathing...",NaN
4955,4955,The patient is a 5-1/2-year-old with Down syn...,Cardiovascular / Pulmonary,Atrioventricular Septal Defect,"HISTORY: ,The patient is a 5-1/2-year-old wit...",NaN
4956,4956,A critically ill 67-year-old with multiple me...,Cardiovascular / Pulmonary,Atrial Flutter - Progress Note,"HISTORY OF PRESENT ILLNESS: , Hospitalist foll...","cardiovascular / pulmonary, rapid ventricular ..."
4957,4957,The patient is a very pleasant 62-year-old Af...,Cardiovascular / Pulmonary,Atrial Fibrillation Management,"REASON FOR CONSULTATION: , Atrial fibrillation...",NaN
4958,4958,Ash split venous port insertion. The right an...,Cardiovascular / Pulmonary,Ash Split Venous Port,"ASH SPLIT VENOUS PORT,PROCEDURE DETAILS: ,The...","cardiovascular / pulmonary, ash split venous p..."


## Stage 1: Extraction

Runs the extraction LLM (`extract_entities`) on a single transcription (a
cardiac surgery operative note) and returns the raw entity spans, grouped
by type, with no ontology constraint applied yet.

In [ ]:
entities = extract_entities(d.iloc[4952]['transcription'])

In [5]:
entities

[RawEntity(entity_type=<EntityType.CONDITION: 'condition'>, text='Aortic valve stenosis', context='DIAGNOSIS: , Aortic valve stenosis with coronary artery disease associated with congestive heart failure.', dose=None, route=None, frequency=None),
 RawEntity(entity_type=<EntityType.CONDITION: 'condition'>, text='coronary artery disease', context='DIAGNOSIS: , Aortic valve stenosis with coronary artery disease associated with congestive heart failure.', dose=None, route=None, frequency=None),
 RawEntity(entity_type=<EntityType.CONDITION: 'condition'>, text='congestive heart failure', context='DIAGNOSIS: , Aortic valve stenosis with coronary artery disease associated with congestive heart failure.', dose=None, route=None, frequency=None),
 RawEntity(entity_type=<EntityType.PAST_MEDICAL_HISTORY: 'past_medical_history'>, text='diabetes', context='The patient has diabetes and is morbidly obese.', dose=None, route=None, frequency=None),
 RawEntity(entity_type=<EntityType.PAST_MEDICAL_HISTORY:

## Stage 2: Ontology candidate retrieval

For each raw entity above, `search_candidates` queries OLS4 against the
ontology (or ontologies) mapped to that entity's type and returns a small
shortlist of plausible codes. This is a live network call per entity,
hence the progress bar.

In [6]:
candidates = []

for ent in tqdm(entities):
    candidates.append(search_candidates(ent.entity_type, ent.text))

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [00:24<00:00,  1.04it/s]


## Stage 3: Grounding

Each entity, together with its Stage 2 shortlist, is sent to Typesafe's Jev
model via `ground_entity`. Jev selects the correct candidate — or
`NO_MATCH` — from that closed list; it never sees the ontology itself.
Results are cached in `grounding_cache.json`, so re-running this cell after
the first pass replays cached answers instead of spending on identical
calls again.

In [7]:
grounding_results = []

for entity, candidate in zip(entities, candidates):
    grounding_results.append(ground_entity(entity, candidate))

## Flatten grounding results for inspection

`GroundedEntity` objects are awkward to scan as-is, so this pulls out the
fields worth comparing (extracted text, matched ontology label, confidence)
into a flat table. A `matched_label` of `NaN` means Typesafe returned
`NO_MATCH` — but check `confidence` alongside it: a `NO_MATCH` with high
confidence is a real, deliberate rejection of all the candidates, while a
`NO_MATCH` with `confidence == 0.0` means Stage 2 found zero candidates and
Typesafe was never even called.

In [9]:
outputs = []

for g in grounding_results:
    outputs.append(
        {
            "type": g.raw.entity_type.value,
            "value": g.raw.text,
            "matched_label": g.label,
            "context": g.raw.context,
            "dose": g.raw.dose,
            "route": g.raw.route,
            "frequency": g.raw.frequency,
            "confidence": g.confidence
        })

In [10]:
pd.DataFrame(outputs)

,type,value,matched_label,context,dose,route,frequency,confidence
0,condition,Aortic valve stenosis,aortic valve stenosis,"DIAGNOSIS: , Aortic valve stenosis with corona...",None,None,None,0.71
1,condition,coronary artery disease,coronary artery disorder,"DIAGNOSIS: , Aortic valve stenosis with corona...",None,None,None,0.99
2,condition,congestive heart failure,congestive heart failure,"DIAGNOSIS: , Aortic valve stenosis with corona...",None,None,None,0.69
3,past_medical_history,diabetes,type 2 diabetes mellitus,The patient has diabetes and is morbidly obese.,None,None,None,0.77
4,past_medical_history,morbidly obese,morbid obesity,The patient has diabetes and is morbidly obese.,None,None,None,0.98
5,procedure,Aortic valve replacement using a mechanical valve,Mechanical Aortic Valve,"PROCEDURES: , Aortic valve replacement using a...",None,None,None,0.57
6,procedure,two-vessel coronary artery bypass grafting pro...,NaN,"PROCEDURES: , Aortic valve replacement using a...",None,None,None,0.00
7,procedure,median sternotomy,NaN,"INCISION: , Median sternotomy,",None,None,None,0.00
8,phenotype,severe congestive heart failure,NaN,"INDICATIONS: , The patient presented with seve...",None,None,None,0.98
9,phenotype,moderately stenotic aortic valve,NaN,The patient was found to have moderately steno...,None,None,None,0.00


## Digging into a specific case

Spot-checks of individual entities, candidate shortlists, and grounded
outputs by index — not necessarily the same entity across cells, just a
quick way to eyeball the extraction, retrieval, and grounding output shapes
independently. Useful for tracing a bad or missing match back to whether it
was a Stage 2 retrieval problem (candidate list is missing the right code)
or a Stage 3 classification problem (the right code was there, but Typesafe
didn't pick it).

In [11]:
entities[3]

RawEntity(entity_type=<EntityType.PAST_MEDICAL_HISTORY: 'past_medical_history'>, text='diabetes', context='The patient has diabetes and is morbidly obese.', dose=None, route=None, frequency=None)

In [12]:
candidates[4]

[OntologyCandidate(ontology='mondo', code='MONDO:1013165', label='morbid obesity, non-human animal', synonyms=[]),
 OntologyCandidate(ontology='mondo', code='MONDO:0005139', label='morbid obesity', synonyms=[]),
 OntologyCandidate(ontology='mondo', code='MONDO:0014309', label='obesity due to CEP19 deficiency', synonyms=['morbid obesity and spermatogenic failure', 'MOSPGF']),
 OntologyCandidate(ontology='mondo', code='MONDO:0013992', label='obesity due to leptin receptor gene deficiency', synonyms=['obesity, morbid, due to leptin receptor deficiency', 'LEPR Deficiency', 'obesity due to leptin receptor gene deficiency']),
 OntologyCandidate(ontology='mondo', code='MONDO:0013991', label='obesity due to congenital leptin deficiency', synonyms=['obesity, morbid, due to leptin deficiency', 'Congenital Leptin Deficiency', 'LEPD', 'leptin deficiency or dysfunction']),
 OntologyCandidate(ontology='mondo', code='MONDO:0019417', label='X-linked intellectual disability-precocious puberty-obesity s

In [13]:
outputs[4]

{'type': 'past_medical_history',
 'value': 'morbidly obese',
 'matched_label': 'morbid obesity',
 'context': 'The patient has diabetes and is morbidly obese.',
 'dose': None,
 'route': None,
 'frequency': None,
 'confidence': 0.98}